<a href="https://colab.research.google.com/github/jolineuichanco/DataAnalytics/blob/main/projects/project-2-prediction-challenge/Project_2_Scoring_Process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Data Analytics for Process Improvement
Author: Joline Uichanco
PROJECT 2: Prediction Challenge
Scoring Process
"""

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. LOAD AND PREPARE TRAINING DATA
# ============================================================
url = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/projects/project-2-prediction-challenge/trainingData.csv'
data = pd.read_csv(url)

# Drop columns not used in modeling
data.drop(columns=["id", "toCoupon_GEQ5min"], inplace=True, errors="ignore")

# Separate target and predictors
outcome = "Y"
predictors = [col for col in data.columns if col != outcome]

# Encode categorical variables (GBM in sklearn requires numeric input)
label_encoders = {}
for col in data.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# ============================================================
# 2. TRAIN / TEST SPLIT (80/20, stratified)
# ============================================================
X = data[predictors]
y = data[outcome]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ============================================================
# 3. TRAIN GBM MODEL (default hyperparameters — baseline)
# ============================================================
gbm_model = GradientBoostingClassifier(random_state=42)
gbm_model.fit(X_train, y_train)

# Evaluate on held-out test set (improvement over original R script)
y_pred_test = gbm_model.predict(X_test)
print("=== Model Performance on Test Set ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_test):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_test):.4f}")
print()
print(classification_report(y_test, y_pred_test, target_names=["Rejected (0)", "Accepted (1)"]))

# ============================================================
# 4. SCORE NEW DATA
# ============================================================

url = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/projects/project-2-prediction-challenge/scoringData.csv'
score_data = pd.read_csv(url)
score_data.drop(columns=["id", "toCoupon_GEQ5min"], inplace=True, errors="ignore")

# Apply the same label encoding to scoring data
for col in score_data.select_dtypes(include=["object"]).columns:
    if col in label_encoders:
        le = label_encoders[col]
        score_data[col] = score_data[col].astype(str).map(
            lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1
        )

scoring = gbm_model.predict(score_data[predictors])
print("=== Scoring Distribution ===")
print(pd.Series(scoring).value_counts().sort_index().rename({0: "Rejected (0)", 1: "Accepted (1)"}))

# ============================================================
# 5. CREATE SUBMISSION FILE
# ============================================================
url = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/projects/project-2-prediction-challenge/teamX_submission.csv'
submission = pd.read_csv(url)
submission["Y"] = scoring
submission.to_csv("teamJU_submission.csv", index=False)
print(f"\nSubmission file saved with {len(submission)} predictions.")

=== Model Performance on Test Set ===
Accuracy : 0.7190
F1 Score : 0.7672

              precision    recall  f1-score   support

Rejected (0)       0.70      0.60      0.65       858
Accepted (1)       0.73      0.81      0.77      1142

    accuracy                           0.72      2000
   macro avg       0.72      0.70      0.71      2000
weighted avg       0.72      0.72      0.72      2000

=== Scoring Distribution ===
Rejected (0)     993
Accepted (1)    1691
Name: count, dtype: int64

Submission file saved with 2684 predictions.
